# Lesson 42: Deploying AI — Gradio, Streamlit and FastAPI 🚀
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Learning goals**
1. Build a shareable AI demo with **Gradio** (Interface, Blocks, ChatInterface)
2. Build a data-style AI web app with **Streamlit** and open it from Colab
3. Serve a model as a **REST API** with **FastAPI**, with validation and auto-generated docs
4. Choose where to host each one, and apply basic security

**⏱️ Timing:** Gradio 20' | Streamlit 15' | FastAPI 15' | hosting & security 10'

> ⚙️ **GPU recommended:** Runtime → Change runtime type → **T4 GPU** (the local LLM runs much faster).

## 🍽️ An analogy: three ways to serve food
| Tool | Like… | You get | Best for |
|---|---|---|---|
| **Gradio** | A **food truck**: quick to set up, people try your dish instantly | A web UI for one model, with a public link | Demos, sharing a model |
| **Streamlit** | A **café**: a whole menu, nice layout | A multi-part web app written in pure Python | Dashboards, data + AI apps |
| **FastAPI** | A **kitchen with a delivery window**: other programs place orders | A REST **API** (JSON in, JSON out) | Powering apps, websites, phones |

All three are free, open-source Python libraries. We'll serve the same two models with each: a **sentiment classifier** (Lesson 20) and the **local chat LLM**.

In [ ]:
# 🦙 Start a free LLM server *inside* Colab (Ollama). No API key or account needed.
# ⏳ First run: about 2–3 minutes (installs Ollama and downloads the model).
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

MODEL = "qwen2.5:3b"     # small and fast. For agent lessons, "qwen2.5:7b" is smarter (slower to download)
!ollama pull {MODEL} > /dev/null 2>&1
print("✅ Ollama is running with", MODEL)

In [ ]:
!pip -q install gradio streamlit fastapi uvicorn openai
import warnings; warnings.filterwarnings("ignore")
from transformers import pipeline
from openai import OpenAI
sentiment = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
print(sentiment("Deployment day is exciting!"))

## Part 1: Gradio 🎛️
### `gr.Interface` — a function becomes an app

In [ ]:
import gradio as gr

def analyse(text):
    r = sentiment(text)[0]
    return {r["label"]: r["score"], ("NEGATIVE" if r["label"] == "POSITIVE" else "POSITIVE"): 1 - r["score"]}

demo = gr.Interface(fn=analyse, inputs=gr.Textbox(label="Your review", lines=3), outputs=gr.Label(label="Mood"),
                    title="🎬 Review Mood Meter", examples=[["Best film ever!"], ["I want my money back."]])
demo.launch(share=True)        # share=True gives a public link for 1 week — anyone with it can use your app!

### `gr.ChatInterface` — a chatbot with streaming

In [ ]:
def study_buddy(message, history):
    messages = [{"role": "system", "content": "You are a friendly study buddy for teenagers. Keep answers short."}]
    for turn in history:                           # Gradio passes the chat so far as OpenAI-style messages
        if isinstance(turn.get("content"), str):       # (skip images or other non-text items)
            messages.append({"role": turn["role"], "content": turn["content"]})
    messages.append({"role": "user", "content": message})
    reply = ""
    for chunk in client.chat.completions.create(model=MODEL, messages=messages, stream=True):
        reply += chunk.choices[0].delta.content or ""
        yield reply                                # yield → the answer appears word by word

demo.close()
chat_app = gr.ChatInterface(study_buddy, title="📚 Study Buddy")
chat_app.launch(share=True)

### `gr.Blocks` — your own layout

In [ ]:
with gr.Blocks(title="AI Toolkit") as toolkit:
    gr.Markdown("# 🧰 AI Toolkit\nTwo tools in one app.")
    with gr.Tab("Mood meter"):
        inp = gr.Textbox(label="Text"); out = gr.Label()
        gr.Button("Analyse").click(analyse, inp, out)
    with gr.Tab("Explain like I'm 10"):
        topic = gr.Textbox(label="Topic"); level = gr.Slider(5, 18, value=10, step=1, label="Age")
        answer = gr.Markdown()
        def explain(t, age):
            return client.chat.completions.create(model=MODEL, messages=[{"role": "user",
                   "content": f"Explain {t} to a {int(age)}-year-old in 3 sentences."}]).choices[0].message.content
        gr.Button("Explain").click(explain, [topic, level], answer)

chat_app.close()
toolkit.launch(share=True)

### 🧩 Exercise 1
1. Add a third tab to the toolkit, e.g. a **quiz generator** with a dropdown for the subject.
2. Send your share link to a classmate. What happens to your app when you close Colab? (It stops! A share link is for **demos**, not permanent hosting.)

## Part 2: Streamlit 📊
A Streamlit app is a Python **script** that re-runs from top to bottom every time the user clicks something. We write it to a file and run it as a server.

In [ ]:
%%writefile app.py
import streamlit as st
from transformers import pipeline
from openai import OpenAI

st.set_page_config(page_title="Class Feedback Analyser", page_icon="📝")
st.title("📝 Class Feedback Analyser")
st.write("Paste student feedback (one comment per line) to see the mood and an AI summary.")

@st.cache_resource                       # load the model ONCE, not on every click
def load_model():
    return pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
sentiment = load_model()
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

text = st.text_area("Feedback", "The experiments were fun\nToo much homework this week\nI finally understand forces!", height=150)
if st.button("Analyse"):
    lines = [l for l in text.split("\n") if l.strip()]
    results = sentiment(lines)
    positive = sum(r["label"] == "POSITIVE" for r in results)
    col1, col2 = st.columns(2)
    col1.metric("😊 Positive", positive)
    col2.metric("😟 Negative", len(lines) - positive)
    st.bar_chart({"positive": [positive], "negative": [len(lines) - positive]})
    st.dataframe([{"comment": l, "mood": r["label"], "confidence": round(r["score"], 2)} for l, r in zip(lines, results)])
    with st.spinner("Writing a summary…"):
        summary = client.chat.completions.create(model="MODEL_NAME", messages=[{"role": "user",
                  "content": "Summarise this class feedback for the teacher in 2 sentences:\n" + "\n".join(lines)}]).choices[0].message.content
    st.success(summary)

In [ ]:
import subprocess, time
open("app.py", "w").write(open("app.py").read().replace("MODEL_NAME", MODEL))      # put the model name into the app
streamlit_proc = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
                                   "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
                                  stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(10)
from google.colab.output import eval_js
print("👉 Open your Streamlit app:", eval_js("google.colab.kernel.proxyPort(8501)"))
print("(This link only works for you, while this notebook is running.)")

### 🧩 Exercise 2
1. Add a `st.sidebar` with a slider that sets the summary length.
2. Add `st.file_uploader` so the teacher can upload a `.txt` file of comments.
3. Why do we need `@st.cache_resource`? Remove it, click Analyse twice, and compare the speed.

## Part 3: FastAPI — an API for other programs 🔌
An **API** lets apps, websites and phones use your model by sending **JSON** over HTTP. FastAPI checks the input with **Pydantic** and writes interactive docs for you.

In [ ]:
%%writefile api.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from transformers import pipeline
from openai import OpenAI
import os

app = FastAPI(title="School AI API", version="1.0")
sentiment = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL = os.environ.get("CHAT_MODEL", "qwen2.5:3b")

class TextIn(BaseModel):
    text: str = Field(min_length=1, max_length=1000, examples=["I love this class!"])

class SentimentOut(BaseModel):
    label: str
    score: float

class ChatIn(BaseModel):
    message: str = Field(min_length=1, max_length=2000)
    age: int = Field(default=14, ge=8, le=19)

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/sentiment", response_model=SentimentOut)
def predict_sentiment(body: TextIn):
    r = sentiment(body.text)[0]
    return SentimentOut(label=r["label"], score=round(r["score"], 4))

@app.post("/explain")
def explain(body: ChatIn):
    if "password" in body.message.lower():
        raise HTTPException(status_code=400, detail="Please don't send passwords.")
    reply = client.chat.completions.create(model=MODEL, messages=[{"role": "user",
            "content": f"Explain this to a {body.age}-year-old in 3 sentences: {body.message}"}])
    return {"answer": reply.choices[0].message.content}

In [ ]:
import os
api_proc = subprocess.Popen(["uvicorn", "api:app", "--host", "127.0.0.1", "--port", "8000"],
                            env={**os.environ, "CHAT_MODEL": MODEL}, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(25)          # the API loads the model when it starts
import requests
BASE = "http://127.0.0.1:8000"
print(requests.get(f"{BASE}/health").json())
print(requests.post(f"{BASE}/sentiment", json={"text": "This lesson was brilliant"}).json())
print(requests.post(f"{BASE}/explain", json={"message": "what is an API?", "age": 12}).json())

In [ ]:
# Validation in action: bad input gets a clear error, not a crash
bad = requests.post(f"{BASE}/sentiment", json={"text": ""})
print(bad.status_code, bad.json()["detail"][0]["msg"])
bad = requests.post(f"{BASE}/explain", json={"message": "hi", "age": 99})
print(bad.status_code, bad.json()["detail"][0]["msg"])

print("\n📖 Interactive API docs:", eval_js("google.colab.kernel.proxyPort(8000)") + "docs")

### 🧩 Exercise 3
1. Open the `/docs` page and try the endpoints with **Try it out**.
2. Add a `/quiz` endpoint that takes a `topic` and returns 3 questions as **JSON** (Lesson 25's JSON mode).
3. Connect them: make the Gradio app call **your API** with `requests` instead of the model directly. That's how real apps separate the **front end** from the **back end**.

## Part 4: Hosting and security 🌍🔐
| To host… | Popular options |
|---|---|
| Gradio or Streamlit demo | **Hugging Face Spaces** (free tier), **Streamlit Community Cloud** |
| FastAPI | Render, Railway, Fly.io, Google Cloud Run, AWS, Azure — usually in a **Docker** container |
| The LLM itself | An API provider (Lesson 25), or your own GPU server running Ollama / vLLM |

**Production checklist**
- [ ] Secrets (API keys) in environment variables or the host's secret settings — never in code or Git
- [ ] **Input validation** (Pydantic) and **length limits**
- [ ] **Rate limiting** so one user can't run up your bill
- [ ] **Guardrails** on inputs and outputs (Lesson 29)
- [ ] **Logging and monitoring** — but don't log personal data you don't need
- [ ] A clear **privacy notice** and an **"AI-generated" label** on outputs

In [ ]:
streamlit_proc.kill(); api_proc.kill(); gr.close_all()
print("🛑 All servers stopped")

### 🚀 Final challenge: ship your capstone
Take your Lesson 16 project (or any project from Lessons 30–40) and deploy it:
1. A **FastAPI** back end with at least 2 endpoints and validation
2. A **Gradio** or **Streamlit** front end that calls it
3. A one-page README: what it does, how to run it, its limits, and how it uses AI responsibly

## 📌 Summary
- **Gradio**: a model → shareable web demo in a few lines (`Interface`, `ChatInterface`, `Blocks`)
- **Streamlit**: a Python script → interactive web app (re-runs on every click; cache models)
- **FastAPI**: a model → a validated REST API with automatic docs, for other programs to use
- Host on Spaces / Streamlit Cloud / a container platform, and follow the security checklist

🎓 **That's the end of the advanced track!** From tensors to deployed agents — well done.